<a href="https://colab.research.google.com/github/satyam-1605/RAG-from-basic-to-advance/blob/main/Rag_with_memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [64]:
%%capture --no-stderr
%pip install --upgrade --quiet langchain langchain-community langchainhub langchain-chroma beautifulsoup4
!pip install -q langchain_google_genai

In [65]:
%pip install --upgrade --quiet langchain_classic

In [66]:
import os
from google.colab import userdata

In [67]:
os.environ["GEMINI_API_KEY"]=userdata.get('GEMINI_API_KEY')

In [68]:

os.environ["LANGSMITH_TRACING"]="true"
os.environ["LANGSMITH_ENDPOINT"]="https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"]=userdata.get('LANGSMITH_API_KEY')
os.environ["LANGSMITH_PROJECT"]="Rag_with_memory"


In [69]:
import warnings
warnings.filterwarnings('ignore')

In [70]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")

In [230]:
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model="gemini-3-flash-preview",convert_system_message_to_human=True)

In [114]:
print(model.invoke("Hi").content)

Hello! How can I help you today?


In [73]:
import bs4


In [74]:
from langchain_classic import hub

In [75]:
from langchain_classic.chains import create_retrieval_chain

In [76]:
from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain,
)

In [77]:

from langchain_chroma import Chroma

In [78]:
from langchain_community.document_loaders import WebBaseLoader

In [79]:
from langchain_core.prompts import ChatPromptTemplate

In [80]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [81]:
from langchain_core.prompts import MessagesPlaceholder

In [82]:
loader = WebBaseLoader(web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),bs_kwargs=dict(parse_only=bs4.SoupStrainer(class_=("post-content","post-title","post-header"))))

In [83]:
doc=loader.load()

In [84]:
text_splitter= RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(doc)

In [85]:
vectorstore= Chroma.from_documents(documents=splits,embedding=embeddings)
retriever = vectorstore.as_retriever()

In [86]:
retriever

VectorStoreRetriever(tags=['Chroma', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x7fbe1021eba0>, search_kwargs={})

In [116]:
system_prompt= (
    "you are an assistant for question-answering task."
    "Use the following pieces of retrieved context to answer the question"
    "If you do'not know the answer, say that you do not know."
    "Use three sentences maximum and keep the answer concise."
    "\n\n"
    "{context}"


)

In [117]:
chat_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),

        ("human","{input}")
    ]
)

In [118]:
question_answering_chain=create_stuff_documents_chain(model,chat_prompt)

In [119]:
rag_chain = create_retrieval_chain(retriever, question_answering_chain)

In [120]:
response = rag_chain.invoke({"input":"what is MRKL?"})

In [92]:
response["answer"]

'MRKL, which stands for “Modular Reasoning, Knowledge and Language,” is a neuro-symbolic architecture designed for autonomous agents. It uses a general-purpose LLM as a router to direct inquiries to specialized "expert" modules, which can be either neural models or symbolic tools like calculators and APIs. This system relies on the LLM’s capability to determine when and how to use these external tools effectively.'

In [121]:
from langchain_classic.chains import create_history_aware_retriever

In [171]:

retriever_prompt = (
    "Given a chat history and the latest user question which might reference "  # fixed 'use' → 'user'
    "context in the chat history, formulate a standalone question which can be "
    "understood without the chat history. Do NOT answer the question, just "
    "reformulate it if needed and otherwise return it as is."
)

In [211]:
def clean_query(query: str) -> str:
    # Remove surrounding quotes and whitespace
    query = query.strip().strip("'\"")
    # If model answered instead of rephrasing, extract first sentence only
    first_sentence = query.split("?")[0] + "?" if "?" in query else query.split("\n")[0]
    return first_sentence.strip()

In [212]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

In [232]:
smart_cleaner_llm = model | StrOutputParser() | RunnableLambda(clean_query)

In [214]:
contextualize_q_prompt= ChatPromptTemplate.from_messages(
    [
        ("system",retriever_prompt),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human","{input}")
    ]
)

In [233]:
history_aware_retriever= create_history_aware_retriever(smart_cleaner_llm, retriever, contextualize_q_prompt)

In [234]:
from langchain_classic.chains import create_retrieval_chain

In [235]:
from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain,
)

In [236]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        # MessagesPlaceholder("chat_history"),
        ("human","{input}")
    ]
)

In [237]:
question_answering_chain=create_stuff_documents_chain(model,qa_prompt)

In [238]:
rag_chain= create_retrieval_chain(history_aware_retriever,question_answering_chain)

In [239]:
from langchain_core.messages import HumanMessage, AIMessage

In [240]:
chat_history=[]

In [241]:
question1= "what is Task Decompostion?"

In [242]:
message1= rag_chain.invoke({"input":question1,"chat_history":chat_history})

In [243]:
message1["answer"]

'Task decomposition is the process of breaking down large, complex tasks into smaller, manageable subgoals to enable more efficient problem-solving. It can be achieved through techniques like Chain of Thought (CoT) prompting, task-specific instructions, or human inputs. Advanced methods like Tree of Thoughts and LLM+P further enhance this by exploring multiple reasoning paths or utilizing external classical planners.'

In [244]:
chat_history.extend(
    [
        HumanMessage(content=question1),
        AIMessage(content=message1["answer"])
    ]
)

In [245]:
chat_history

[HumanMessage(content='what is Task Decompostion?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Task decomposition is the process of breaking down large, complex tasks into smaller, manageable subgoals to enable more efficient problem-solving. It can be achieved through techniques like Chain of Thought (CoT) prompting, task-specific instructions, or human inputs. Advanced methods like Tree of Thoughts and LLM+P further enhance this by exploring multiple reasoning paths or utilizing external classical planners.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [246]:
second_question = "what are common ways of doing it?"

In [248]:
message2= rag_chain.invoke({"input":"what are common ways of doing it?" ,"chat_history":chat_history})

In [249]:
message2["answer"]

'Common ways of task decomposition include simple prompting (e.g., "think step by step"), task-specific instructions, human inputs, and advanced techniques like Chain of Thought (CoT) or Tree of Thoughts (ToT). Another approach, LLM+P, involves outsourcing planning to an external classical planner using Planning Domain Definition Language (PDDL) as an interface. Additionally, systems like HuggingGPT use few-shot prompting to parse requests into structured tasks with specific attributes and dependencies.'

In [250]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [251]:
store={}

In [252]:
def get_session_history(session_id: str) -> BaseChatMessageHistory:
  if session_id not in store:
    store[session_id]= ChatMessageHistory()
  return store[session_id]

In [253]:
conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    output_messages_key="answer",
    history_messages_key="chat_history"
)



In [254]:
conversational_rag_chain.invoke(
    {"input":"What is Task Decompostion?"},
    config={
        "configurable": {"session_id":"abc123"}
    },
    )["answer"]


"Task decomposition is the process of breaking down a complex task into smaller, simpler, and more manageable steps to facilitate planning. It can be achieved through LLM prompting techniques like Chain of Thought or Tree of Thoughts, task-specific instructions, or human inputs. This method transforms large problems into multiple subgoals or thought steps, providing insight into the model's thinking process."

In [255]:
store

{'abc123': InMemoryChatMessageHistory(messages=[HumanMessage(content='What is Task Decompostion?', additional_kwargs={}, response_metadata={}), AIMessage(content="Task decomposition is the process of breaking down a complex task into smaller, simpler, and more manageable steps to facilitate planning. It can be achieved through LLM prompting techniques like Chain of Thought or Tree of Thoughts, task-specific instructions, or human inputs. This method transforms large problems into multiple subgoals or thought steps, providing insight into the model's thinking process.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])}

In [256]:
conversational_rag_chain.invoke(
    {"input":"what are common ways of doing it?"},
    config={
        "configurable": {"session_id":"abc123"}
    },
)["answer"]

'Task decomposition can be performed using simple prompting, task-specific instructions, or human inputs. Standard techniques include Chain of Thought (CoT) to "think step by step" and Tree of Thoughts (ToT), which explores multiple reasoning possibilities at each step. Additionally, the LLM+P approach involves outsourcing planning to an external classical planner using Planning Domain Definition Language (PDDL).'

In [257]:
for message in store["abc123"].messages:
  if isinstance(message,AIMessage):
    prefix="AI"
  else:
    prefix="User"
  print(f"{prefix}: {message.content}\n")

User: What is Task Decompostion?

AI: Task decomposition is the process of breaking down a complex task into smaller, simpler, and more manageable steps to facilitate planning. It can be achieved through LLM prompting techniques like Chain of Thought or Tree of Thoughts, task-specific instructions, or human inputs. This method transforms large problems into multiple subgoals or thought steps, providing insight into the model's thinking process.

User: what are common ways of doing it?

AI: Task decomposition can be performed using simple prompting, task-specific instructions, or human inputs. Standard techniques include Chain of Thought (CoT) to "think step by step" and Tree of Thoughts (ToT), which explores multiple reasoning possibilities at each step. Additionally, the LLM+P approach involves outsourcing planning to an external classical planner using Planning Domain Definition Language (PDDL).



In [258]:
conversational_rag_chain.invoke(
    {
      "input":"What is a prompt technique like step xyz?"
    },
    config={"configurable":{"session_id":"abc123"}},
)["answer"]



'The prompt technique using "Steps for XYZ" is **task decomposition**, which breaks down complex tasks into smaller, manageable subgoals. It can be performed by an LLM using simple prompts, task-specific instructions, or human inputs. This approach is similar to Chain of Thought (CoT) prompting, which instructs the model to "think step by step."'

In [259]:
store

{'abc123': InMemoryChatMessageHistory(messages=[HumanMessage(content='What is Task Decompostion?', additional_kwargs={}, response_metadata={}), AIMessage(content="Task decomposition is the process of breaking down a complex task into smaller, simpler, and more manageable steps to facilitate planning. It can be achieved through LLM prompting techniques like Chain of Thought or Tree of Thoughts, task-specific instructions, or human inputs. This method transforms large problems into multiple subgoals or thought steps, providing insight into the model's thinking process.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='what are common ways of doing it?', additional_kwargs={}, response_metadata={}), AIMessage(content='Task decomposition can be performed using simple prompting, task-specific instructions, or human inputs. Standard techniques include Chain of Thought (CoT) to "think step by step" and Tree of Thoughts (ToT), which explo

In [260]:
for message in store["abc123"].messages:
  if isinstance(message,AIMessage):
    prefix="AI"
  else:
    prefix="User"
  print(f"{prefix}: {message.content}\n")

User: What is Task Decompostion?

AI: Task decomposition is the process of breaking down a complex task into smaller, simpler, and more manageable steps to facilitate planning. It can be achieved through LLM prompting techniques like Chain of Thought or Tree of Thoughts, task-specific instructions, or human inputs. This method transforms large problems into multiple subgoals or thought steps, providing insight into the model's thinking process.

User: what are common ways of doing it?

AI: Task decomposition can be performed using simple prompting, task-specific instructions, or human inputs. Standard techniques include Chain of Thought (CoT) to "think step by step" and Tree of Thoughts (ToT), which explores multiple reasoning possibilities at each step. Additionally, the LLM+P approach involves outsourcing planning to an external classical planner using Planning Domain Definition Language (PDDL).

User: What is a prompt technique like step xyz?

AI: The prompt technique using "Steps f